# IITB Insti-Assist — A RAG-Powered AI Assistant for IIT Bombay
### Scope: **Academic Assistant** (course registration, grading policy, academic calendar, exam rules)

This notebook builds an end-to-end Retrieval-Augmented Generation (RAG) system that answers questions about **IIT Bombay's academic rules** like grading, SPI/CPI, registration, the academic calendar, academic malpractice penalties, and medals/prizes are grounded entirely in real IIT Bombay Academic Office documents.

**Pipeline:** Data ingestion → Chunking → Embedding (sentence-transformers) → Vector search (FAISS) → Grounded generation (Gemini API) → Source-cited answer → Gradio chat UI.

The assistant is instructed to **say "I don't know" whenever the retrieved context does not support an answer** as it will not fall back on the LLM's general knowledge about IIT Bombay.


## 1. Setup

In [14]:
!pip install -q sentence-transformers faiss-cpu google-generativeai gradio numpy

import importlib
for pkg in ["sentence_transformers", "faiss", "google.generativeai", "gradio", "numpy"]:
    try:
        importlib.import_module(pkg)
        print(f"[ok] {pkg} already installed")
    except ImportError:
        print(f"[missing] {pkg} -- installing...")
        import subprocess, sys as _sys
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q",
                        {"faiss": "faiss-cpu", "sentence_transformers": "sentence-transformers", "google.generativeai": "google-generativeai"}.get(pkg, pkg)])


[ok] sentence_transformers already installed
[ok] faiss already installed
[ok] google.generativeai already installed
[ok] gradio already installed
[ok] numpy already installed


In [15]:
import os
import re
import json
import textwrap
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Any

from sentence_transformers import SentenceTransformer
import faiss

print("Imports OK")


Imports OK


## 2. Data

as Per the project's functional requirements, this assistant is grounded in **5 real IIT Bombay Academic Office documents** (text below is transcribed/condensed from the official PDFs, with source URLs kept for citation and for the "known limitations" discussion):

| # | Document |
|---|----------|
| 1 | UG Rules & Regulations |
| 2 | Academic Calendar 2025–2026 |
| 3 | Academic Calendar 2026–2027 |
| 4 | Disciplinary Actions for Academic Malpractice |
| 5 | Rules for Award of Medals & Academic Prizes (UG & PG) |

These are shipped as embedded text below so the notebook is fully self-contained and doesn't require internet access to run the RAG pipeline itself.


In [16]:
"""
Real source documents for IITB Insti-Assist (Academic Assistant scope).
All text below is copied from official IIT Bombay Academic Office documents
(publicly available PDFs), condensed slightly to remove page-break artifacts
and repeated headers, but the content and wording is preserved from source.
"""

DOC_UG_RULEBOOK = """IIT BOMBAY RULES & REGULATIONS FOR UNDERGRADUATE PROGRAMMES (B.Tech./B.S./B.Des./Dual Degree), Updated June 2025.

1. CURRICULUM AND CREDITS
IIT Bombay follows a credit-based semester system with two regular semesters (Autumn: July-Nov;
Spring: Jan-Apr/May) plus an optional Summer Term. Credit for a course = 2xL + 2xT + P, where L, T,
P are weekly lecture, tutorial and practical hours. Minimum credits required for a B.Tech./B.S./B.Des.
degree are generally between 266 and 282 (for batches till 2021), split roughly into Basic Sciences
(60-62), Engineering Sciences & Skills (30), HSS Electives (12), Institute Electives (12), and
Departmental courses (152-168). A Dual Degree (B.Tech.+M.Tech.) additionally requires 24 credits
of Honours-equivalent work, 24 credits of masters-level courses, and 72 credits of a masters thesis
(spread over 14 months).

MINOR AND HONOURS: A Minor requires 30 credits of additional coursework in another discipline;
allotment is based on highest CPI among applicants, and full completion within the programme
duration is required or the minor is not awarded. Honours requires 24 additional credits in the
student's own discipline. Students may pursue more than one minor, or a minor plus honours,
subject to timetable and workload constraints.

2. FACULTY ADVISER
Every student is assigned a Faculty Adviser on joining. The Adviser approves course registration,
advises on course load appropriate to academic standing, guides students through course
adjustment/drop decisions, and recommends semester drops to the DUGC/UGAPEC when needed.

3. REGISTRATION
Registration each semester is mandatory and done online through the Course Registration Form
(CRF) on the ASC portal, approved by the Faculty Adviser. Late registration is allowed only for valid
reasons and requires a late fee, by the date fixed in the Academic Calendar. Students with
outstanding dues to the Institute or a hostel cannot register. From the third semester onward,
registration load depends on the student's Academic Standing category (see Section 5). Institute
Elective courses must be from a different Academic Unit than the student's own and must not be a
Core/Departmental Elective course in the student's curriculum. NCC/NSO/NSS (one is mandatory)
is graded Pass(PP)/Not-Pass(NP) based on 80% minimum attendance.

COURSE ADJUSTMENT/DROP: Adjustments (add/delete courses) are permitted up to a date fixed in
the Academic Calendar (typically about one week after semester start), subject to credit limits.
Courses can be dropped later (typically 20-30 days after the mid-semester exam) provided the
minimum 18-credit requirement is still met.

DROPPING A SEMESTER: Allowed with prior DUGC/UGAPEC approval for severe health issues (needs
an IIT Hospital medical certificate or CMO-authenticated certificate) or other valid unavoidable
reasons. Not permitted after a DX/II grade or semester-end exam has already been awarded, and
generally not for more than one continuous year. Hostel facilities are not extended for a semester
in which the student has not registered.

SUMMER TERM: Courses run at accelerated pace with the same total contact hours; minimum 5
students must register (except Minor courses, minimum 1). A student may take up to 24 credits of
Mandatory Courses in summer term in which they have an FR/DX/DR/W grade; NO restriction
applies after the regular programme duration expires.

4. ACADEMIC STANDING AND REGISTRATION LOAD
Academic Standing categories (based on CPI and outstanding FR/DX/DR/W grades in core courses,
computed from the two preceding regular semesters):
- Category I: CPI >= 8, no outstanding FR/DX/DR/W in a core course. Max 54 credits/semester.
- Category II: CPI < 8, no outstanding FR/DX/DR/W in a core course. Max 48 credits/semester.
- Category III: at least one outstanding FR/DX/DR/W in core + at most one FR/DX elsewhere
  (with >=18 credits earned each of the last two semesters). Max 48 credits/semester.
- Category IV: at least one outstanding FR/DX/DR/W in core + MORE than one FR/DX elsewhere.
  Max 48 credits/semester.
- Category V: did not earn at least 18 credits in at least one of the last two regular semesters.
  Max 36 credits/semester.
- Category VI (ARP - Academic Rehabilitation Programme): accumulated FR/DX worth 36+ credits
  in core courses (supersedes all other categories). Max 24 credits/semester.
Every student must register for a minimum of 18 credits each semester (unless nearing completion
of the degree). Category VI students are guided by a dedicated ARP Faculty Adviser and are
transferred back to regular status once FR/DX credits fall below 36; failure to exit ARP within three
consecutive semesters may lead to termination of registration.

AUDIT COURSES: A UG student may audit a maximum of TWO courses in the entire programme
(only Category I/II students, during a regular semester, with instructor permission). Audited
courses earn an 'AU' grade (zero grade points, not counted in SPI/CPI) and cannot count toward
Minor/Honours.

5. EXAMINATION AND GRADING
Theory courses: in-semester assessment (quizzes, tests, assignments, viva) typically carries 50-60%
weightage, including one mid-semester exam of ~25-30% weightage; end-semester exam typically
carries 40-50% and is of about 3 hours covering the full syllabus. Laboratory courses: in-semester
work ~75%, semester-end test ~25%.

GRADING SCALE (letter grade -> grade point):
AP = 10 (exceptional performance, awarded only in courses with >50 registered students, max 2%
of class strength), AA = 10, AB = 9, BB = 8, BC = 7, CC = 6, CD = 5, DD = 4 (minimum passing grade),
FR = 0 (Fail, repeat course - for mandatory/core courses), FF = 0 (Fail, eligible for one
re-examination at max grade DD), DX = 0-equivalent (Dropped due to inadequate attendance,
<80%; treated as a course drop for CPI purposes), W = Course Withdrawn, AU = Satisfactory in an
audit course, PP = Passed (non-credit activities), NP = Not Passed, II = Incomplete (placeholder,
converted to a regular grade after the make-up exam, or auto-converted to FR if not resolved by
next semester), DR = Dropped (placeholder for a cleared drop).

A student PASSES a course with any grade from AP to DD (or AU for audit courses) and FAILS with
FR and/or DX. First-year UG students get an 'FF' (not direct 'FR') on their first failing attempt in a
course; a subsequent attempt yields a regular DD/FR grade.

SPI (Semester Performance Index) = weighted average of grade points, weighted by course credits,
for all courses registered in that semester: SPI = (C1*g1 + C2*g2 + ... ) / (C1 + C2 + ...). Failed
(FR) courses are counted with a grade point of zero in that semester's SPI.
CPI (Cumulative Performance Index) = same calculation across ALL courses counted toward the
degree since the student joined, recalculated at the end of every semester to two decimal places. If
a failed course is cleared later, the CPI reflects only the new passing grade, not the old fail.
Both SPI and CPI are officially stated as NOT convertible into a percentage.

RE-EXAMINATION / MAKE-UP: Valid reasons for a semester-end make-up exam include serious
illness/accident of the student or of a parent/guardian (needs a medical certificate). There is no
provision for re-evaluation of answer scripts, but a re-totaling of marks can be requested for a fee
of about Rs. 200 per course; changes are made only for totaling/tabulation errors.

6. CHANGE OF BRANCH
Regular Branch Change (to any JoSAA-administered branch) was discontinued by the Senate's 256th
meeting for students admitted via JEE (Advanced) 2023 onward. However, conversion to the Liberal
Arts, Sciences and Engineering (LASE) programme via a form of branch change remains available
(from AY 2022-23), based on an "eligibility-CPI" of at least 7.0 computed from specific first-year
courses, plus a written test/interview. All branch-change decisions are final and not reversible, and
can happen only once, at the start of the second year.

7. PERFORMANCE REQUIREMENTS AND ARP
Degree conferral requires passing all prescribed courses, completing NSS/NSO/NCC and other
non-credit requirements, clearing all Institute dues, and having no pending disciplinary case.
Students with poor academic performance are moved to the Academic Rehabilitation Programme
(ARP) once FR/DX credits in core courses reach 36; ARP has its own reduced credit load (24
credits/semester) and dedicated faculty support.

8. UNDERGRADUATE RESEARCH AWARDS (URA)
URA01 (preliminary research recognition, no credits, grade PP, awardable once), URA02 (for
exceptional B.Tech. Project work, +6 credits at grade AA, on top of the BTP grade already given),
and URA03 (equivalent recognition for exceptional Dual Degree Project work) are distinct research
recognitions layered on top of normal project grading.

9. NPTEL/SWAYAM AND SEMESTER EXCHANGE
Students can take up to 12 credits of NPTEL/SWAYAM courses (24 for ARP/extension-year students)
in lieu of departmental/institute electives. A 12-week NPTEL/SWAYAM course is typically equivalent
to 6 credits (sometimes 8, with DUGC/DPGC approval); an 8-week course is typically equivalent to 3
or 4 credits. Grades from such external courses do not count toward SPI/CPI (only credits count);
the transcript records the course and grade as-is.
"""

DOC_ACAD_CALENDAR_2025_26 = """IIT BOMBAY ACADEMIC CALENDAR 2025-2026 (KEY DATES)

AUTUMN SEMESTER 2025 (1st Semester)
- New entrant UG orientation & registration: 21-25 July 2025; on-roll UG/PG registration: 22-24 July 2025.
- Autumn Semester lectures start and end dates: 28 July 2025 (Monday) to 7 November 2025 (Friday).
- Last date for late registration (with fine): 4 August 2025.
- Last date for course adjustment: 4 August 2025.
- Dropping first/full-semester courses without "W" grade: 5-11 August 2025.
- Mid-semester examination for full-semester courses / End-semester exam for first-half-semester courses: 13-21 September 2025.
- Award of DX grades for full-semester and second-half-semester courses: 5-6 November 2025.
- Last date of instruction/lectures: 7 November 2025.
- End-semester examination: 10-22 November 2025.
- On-line submission of grades for full-semester and second-half-semester courses: 10 November - 1 December 2025.
- Date for auto conversion of "FF" and "II" grades to "FR": 9 December 2025.
- Last date for end of academic activities for Autumn Semester: 9 December 2025.
- Calculation of SPI/CPI: 10 December 2025.
- Last date for transfer to Academic Rehabilitation Programme (ARP)/Academic Probation: 10 December 2025.
- Winter vacation (for students): 30 November 2025 - 29 December 2025.

SPRING SEMESTER 2026 (2nd Semester)
- Semester Registration: 30 December 2025 - 1 January 2026; Spring Semester lectures: 5 January 2026 - 17 April 2026.
- Last date for late registration (with fine): 12 January 2026.
- Last date for course adjustment: 12 January 2026.
- Mid-semester examination for full-semester courses / End-semester exam for first-half-semester courses: 21 February - 1 March 2026.
- Last date of instruction/lectures: 17 April 2026.
- End-semester examination: 20 April - 1 May 2026.
- Online submission of grades for full-semester & second-half-semester courses: 20 April - 11 May 2026.
- Date for auto conversion of "FF" and "II" grades to "FR": 16 May 2026.
- Last date for end of academic activities for Spring Semester: 16 May 2026.
- Calculation of SPI/CPI: 16 May 2026.
- Summer vacation (for students): 10 May 2026 - 14 July 2026.

SUMMER TERM 2026
- Registration: 18-20 May 2026; Instruction begins: 21 May 2026; Last date of instruction: 13 July 2026.
- Term-end final examination: 14-18 July 2026.
- On-line submission of grades: 14-21 July 2026.
- Last date of academic activities: 21 July 2026.

NOTES
- Saturdays are included in both mid-semester and end-semester examination date ranges.
- Classes missed due to public holidays may be compensated on Saturdays/Sundays with intimation to the Dean (Academic Programmes), subject to classroom availability.
- Senate meetings (tentative, usually Wednesdays) for 2025-26: 13 Aug 2025, 12 Nov 2025, 18 Feb 2026, 22 Apr 2026.
"""

DOC_ACAD_CALENDAR_2026_27 = """IIT BOMBAY ACADEMIC CALENDAR 2026-2027 (KEY DATES)

AUTUMN SEMESTER 2026
- New entrant UG reporting: 20 July 2026; UG registration: 21-25 July 2026; on-roll registration: 21-24 July 2026.
- Autumn Semester lectures start and end dates: 27 July 2026 (Monday) to 6 November 2026 (Friday).
- Last date for late registration (with fine): 3 August 2026.
- Last date for course adjustment: 3 August 2026.
- Mid-semester examination for full-semester courses / End-semester exam for first-half-semester courses: 12-20 September 2026.
- Last date of instruction/lectures: 6 November 2026.
- End-semester examination: 9-21 November 2026.
- On-line submission of grades for full-semester and second-half-semester courses: 9-30 November 2026.
- Date for auto conversion of "FF" and "II" grades to "FR": 9 December 2026.
- Last date for end of academic activities for Autumn Semester: 9 December 2026.
- Calculation of SPI/CPI: 10 December 2026.
- Winter vacation: 29 November 2026 - 28 December 2026.

SPRING SEMESTER 2027
- Semester Registration: 29 December 2026 - 1 January 2027; Spring Semester lectures: 4 January 2027 - 16 April 2027.
- Last date for late registration (with fine): 11 January 2027.
- Last date for course adjustment: 11 January 2027.
- Mid-semester examination for full-semester courses / End-semester exam for first-half-semester courses: 20-28 February 2027.
- Last date of instruction/lectures: 16 April 2027.
- End-semester examination: 19 April - 1 May 2027.
- Online submission of grades for full-semester & second-half-semester courses: 19 April - 10 May 2027.
- Date for auto conversion of "FF" and "II" grades to "FR": 18 May 2027.
- Last date for end of academic activities for Spring Semester: 18 May 2027.
- Summer vacation: 8 May 2027 - 15 July 2027.

SUMMER TERM 2027
- Registration: 15-20 May 2027; Instruction begins: 21 May 2027; Last date of instruction: 13 July 2027.
- Term-end final examination: 14-18 July 2027.
- Last date of academic activities: 21 July 2027.

SPECIAL ACADEMIC EVENTS
- Teachers' Day: 5 September 2026. National Education Day: 11 November 2026.
- Spring Session of 65th Convocation: 27 February 2027. Commencement 2027 (65th Convocation): 1 May 2027.
- Instructions note that graduation requirements completed 27 Feb 2026 to 5 Aug 2026 (tentative) fall under the 64th Convocation (August 2026); requirements completed 6 Aug 2026 to the Feb 2027 Senate meeting fall under the Spring Session of the 65th Convocation.
"""

DOC_ACADEMIC_MALPRACTICE = """IIT BOMBAY - DISCIPLINARY ACTIONS FOR ACTS OF ACADEMIC MALPRACTICE (Academic Office, effective 16 April 2015)

Disciplinary actions are decided by the D-ADAC for acts attracting up to a DX/FR grade, and by the ADAC for more serious acts.

1. Impersonation / Forging signatures
- Signing attendance for another student: DX grade to the student who signed for another.
- Leaving class after giving attendance: first violation gets a warning from the instructor; a second violation results in an FR grade.
- Impersonating another student during an exam: suspension for one semester.
- Tampering with official documents (grade sheets, medical certificates, etc.): FR grade in the concerned course plus suspension for one semester.
- Forging signatures of faculty/staff: suspension for one year.

2. Copying in home/programming assignments and lab projects
- A zero in the assignment/project plus a one-grade penalty in the course, applied to both the student who copied and the student copied from (unless the material was obtained through hacking/nefarious means, in which case the source student may not be punished).

3. Copying in examinations
- Verbal communication during an exam, with only an invigilator's note as evidence: loss of one grade. With stronger evidence (e.g. identical answers): FR grade.
- Passing chits or unauthorised material to other students: FR grade.
- Mobile phone found in possession (not used) after the exam has begun: loss of one grade. Mobile phone actually used during the exam: FR grade.
- Carrying unauthorised material (chits, tablets, calculators other than permitted, using internet, body scribblings): FR grade -- mere possession/detection is sufficient to attract the penalty.
- Detected copying during evaluation of answer scripts (student A copied from student B): FR grade for both students.
- Making changes to already-valued answer scripts: FR grade.
- Using a mobile phone, chits, books, or other unauthorised material during toilet breaks while the exam is in progress: FR grade plus suspension for one semester.

4. Repeat offences: reported to the ADAC; the standard action is suspension for one semester.

5. Plagiarism in internal reports
- Seminar reports: penalty ranges from a warning (if unintentional) to loss of two grades.
- Project reports: minimum penalty is loss of one grade; maximum penalty is an FR grade.
- More serious cases (falsifying experimental results, falsely claiming original content in an M.Tech/M.Phil/Ph.D thesis): referred to the ADAC, minimum penalty is suspension for one year; in exceptional cases referred to the Apex Committee, which may recommend termination of registration (with or without an exit degree) or, for a Ph.D thesis already awarded, withdrawal of the degree.

6. Indiscipline by TAs
- A TA found aiding/abetting cheating (e.g. inflating marks, tampering with answer scripts) for students they are TAing: suspension for one semester with no stipend during that period.
"""

DOC_MEDALS_AND_PRIZES = """IIT BOMBAY - RULES FOR AWARD OF MEDALS AND ACADEMIC PRIZES FOR UG & PG (Approved 260th Senate, 14 February 2024)

1. President of India Medal (PGM): awarded to the B.S./B.Tech. (with honours) or B.Tech.+M.Tech. Dual Degree/IDDDP student (excluding those without honours) with the highest CPI at the end of the 4th year, based only on the mandatory credit requirements for B.S./B.Tech. (with honours), calculated after final course retagging. Eligibility requires an overall CPI (including honours) of 9.0 or above. Ties are broken using CPI across ALL courses (including additional/minor courses) up to the end of the 4th year. An "II" and/or "W" grade is not a bar, but disqualification occurs for: any "DX"/"FF"/"FR" grade in any course (any tag), extension for final-stage MTPs, completing the programme beyond its prescribed duration, or any DAC/ADAC punishment during the programme.

2. Institute Gold Medal (IGM): given, in the same year a B.Tech./B.S. (with honours) student wins the PGM, to the most outstanding Dual Degree/IDDDP student (and vice versa). Discontinued for students admitted from 2022-23 onward; students admitted till 2021-22 remain eligible. Same eligibility (CPI 9.0+) and disqualification rules as the PGM.

3. Institute Silver Medal (ISM): awarded per academic unit in B.Tech.(honours), Dual Degree, B.S.(honours), B.Des., M.Sc., M.Tech., MPP, M.Des., MBA, MS/MA by Research, and LASE programmes, to the student who qualifies all credit and honours requirements, provided overall CPI is 9.0 or above (or CPI+honours CPI 9.0+ for B.Tech./B.S.). If the graduating strength in a department/programme is below 30, departments are clubbed for consideration; below 30 after clubbing still yields one medal; 40 or more graduating students in clubbed departments yields two medals (maximum two per clubbed group). Exit degrees (e.g. BSc Engg. from B.Tech., M.Tech./MPP exit from a Dual PhD programme) are NOT eligible for medals. Same disqualification rules as PGM/IGM apply (DX/FF/FR in any course, MTP extension, overrun duration, DAC/ADAC punishment).

4. Dr. Shankar Dayal Sharma Gold Medal: awarded to the most outstanding graduating student overall, considering general proficiency, academic excellence, extracurricular activities, and social service, based on overall CPI; same disqualification rules apply.

5. Donor's Medals: a set of named medals (e.g. Miss Jayati Deshmukh Memorial Gold Medal for CSE, Vidyasagar Nehra Gold Medal and Prof Madhav Kulkarni Gold Medal for Civil Engineering, Hindi Vidya Bhavan Gold Medal for the SJMSOM MBA programme, Sharad Maloo Gold Medal, Abhijeet Banerjee SJMSOM Silver Medal) awarded to students who are also selected for the Institute Silver Medal or an equivalent top rank in their programme.

6. Academic Prizes for On-Roll Students: based on weighted SPI average of the previous academic year (weighted by semester credits). 2nd-year UG students (JEE-admitted): top 20 weighted-SPI students get prizes (1st prize Rs 3000, 2nd prize Rs 2000, additional prize Rs 2000). 3rd/4th-year UG, 2nd-4th-year B.Des., and 5th-year Dual Degree students: top 2 weighted-SPI students get prizes if class strength >= 25 (1st and 2nd prize), or just the 1st prize if class strength < 25 (1st prize Rs 3000, 2nd prize Rs 2000). 2nd-year M.Sc./M.Tech./M.Des./M.Phil/MBA/MA-by-Research/MS-by-Research/MPP students: same top-2 structure. A minimum weighted SPI average of 9 is required, and the academic unit must have a class strength of at least 5 students to be eligible for the prize.
"""

CORPUS = [
    {
        "doc_id": "ug_rulebook",
        "title": "UG Rules & Regulations (Academic Office, updated June 2025)",
        "url": "https://acad.iitb.ac.in/files/UG_RULE_BOOK.pdf",
        "text": DOC_UG_RULEBOOK,
    },
    {
        "doc_id": "acad_calendar_2025_26",
        "title": "Academic Calendar 2025-2026",
        "url": "https://acad.iitb.ac.in/sites/default/files/Academic%20Calendar%202025-26_FINAL.pdf",
        "text": DOC_ACAD_CALENDAR_2025_26,
    },
    {
        "doc_id": "acad_calendar_2026_27",
        "title": "Academic Calendar 2026-2027",
        "url": "https://acad.iitb.ac.in/sites/default/files/Academic_Calendar_2026-27_FINAL.pdf",
        "text": DOC_ACAD_CALENDAR_2026_27,
    },
    {
        "doc_id": "academic_malpractice",
        "title": "Disciplinary Actions for Academic Malpractice",
        "url": "http://www.iitb.ac.in/newacadhome/punishments201521July.pdf",
        "text": DOC_ACADEMIC_MALPRACTICE,
    },
    {
        "doc_id": "medals_and_prizes",
        "title": "Rules for Award of Medals and Academic Prizes (UG & PG)",
        "url": "https://www.iitb.ac.in/newacadhome/RulesforAwardofMedalsandAcademicprizesforUGandPG.pdf",
        "text": DOC_MEDALS_AND_PRIZES,
    },
]


In [17]:
print(f"Loaded {len(CORPUS)} source documents:")
for d in CORPUS:
    print(f" - {d['doc_id']:24s} | {len(d['text'].split()):5d} words | {d['title']}")


Loaded 5 source documents:
 - ug_rulebook              |  1431 words | UG Rules & Regulations (Academic Office, updated June 2025)
 - acad_calendar_2025_26    |   439 words | Academic Calendar 2025-2026
 - acad_calendar_2026_27    |   392 words | Academic Calendar 2026-2027
 - academic_malpractice     |   459 words | Disciplinary Actions for Academic Malpractice
 - medals_and_prizes        |   533 words | Rules for Award of Medals and Academic Prizes (UG & PG)


## 3. Chunking strategy

**Why i used paragraph-based chunking with overlap?**
Institute rule documents like the UG Rule Book are organised into numbered sections and paragraphs, each covering one self-contained rule or procedure. Splitting on paragraph boundaries keeps each chunk semantically coherent — a rule and its qualifying conditions stay together and instead of arbitrarily cutting a rule in half at a fixed character count.

I used:
- ***Split unit:*** paragraphs (blank-line separated), then further split any paragraph longer than MAX_CHUNK_WORDS words at sentence boundaries.
- ***Overlap:*** the last 1 sentence of a chunk is repeated at the start of the next chunk when a paragraph had to be split, so a rule that spans a split boundary isn't orphaned.
- ***Chunk size target:*** ~80–160 words, a small enough for precise retrieval, large enough to keep a full rule intact.


In [18]:
MAX_CHUNK_WORDS = 160
MIN_CHUNK_WORDS = 25

def split_sentences(text: str) -> List[str]:
    text = text.replace("\n", " ")
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"])', text)
    return [s.strip() for s in sentences if s.strip()]

def chunk_document(doc: Dict[str, Any]) -> List[Dict[str, Any]]:
    paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
    chunks = []
    buffer_sentences: List[str] = []

    def flush():
        if buffer_sentences:
            chunk_text = " ".join(buffer_sentences).strip()
            if len(chunk_text.split()) >= 5:
                chunks.append(chunk_text)
            buffer_sentences.clear()

    for para in paragraphs:
        para_words = len(para.split())
        if para_words <= MAX_CHUNK_WORDS:
            current_words = sum(len(s.split()) for s in buffer_sentences)
            if current_words + para_words > MAX_CHUNK_WORDS and current_words >= MIN_CHUNK_WORDS:
                flush()
            buffer_sentences.append(para)
        else:
            flush()
            sentences = split_sentences(para)
            cur: List[str] = []
            cur_words = 0
            for sent in sentences:
                sw = len(sent.split())
                if cur_words + sw > MAX_CHUNK_WORDS and cur:
                    chunks.append(" ".join(cur))
                    cur = [cur[-1], sent]
                    cur_words = len(cur[-2].split()) + sw
                else:
                    cur.append(sent)
                    cur_words += sw
            if cur:
                chunks.append(" ".join(cur))
    flush()

    return [
        {
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "url": doc["url"],
            "chunk_id": f"{doc['doc_id']}::{i}",
            "text": c,
        }
        for i, c in enumerate(chunks)
    ]

all_chunks: List[Dict[str, Any]] = []
for doc in CORPUS:
    all_chunks.extend(chunk_document(doc))

print(f"Total chunks: {len(all_chunks)}")
word_counts = [len(c["text"].split()) for c in all_chunks]
print(f"Chunk word count: min={min(word_counts)}, max={max(word_counts)}, avg={sum(word_counts)/len(word_counts):.1f}")
print()


Total chunks: 28
Chunk word count: min=7, max=188, avg=116.3



## 4. Embedding

I used sentence-transformers/all-MiniLM-L6-v2 fast on CPU, and strong enough for short-document semantic search like this. Embeddings are L2-normalized so that inner product search in FAISS is equivalent to cosine similarity.


In [19]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in all_chunks]
embeddings = embed_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print("Embedding matrix shape:", embeddings.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (28, 384)


## 5. Vector index (FAISS)

i used `IndexFlatIP` since our data is small and there is no need for approximate search at this scale.


In [20]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print(f"FAISS index built: {index.ntotal} vectors of dimension {dimension}")


FAISS index built: 28 vectors of dimension 384


## 6. Retrieval

retrieve(query, k) embeds the query with the same model, searches the FAISS index, and returns the top-k chunks along with their similarity scores. A **similarity threshold** (MIN_SIMILARITY) is used downstream to decide whether we even have grounds to attempt an answer, versus saying "I don't know" outright.


In [21]:
MIN_SIMILARITY = 0.30 

def retrieve(query: str, k: int = 4) -> List[Dict[str, Any]]:
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        chunk = dict(all_chunks[idx])
        chunk["score"] = float(score)
        results.append(chunk)
    return results

for r in retrieve("What is the minimum passing grade at IIT Bombay?", k=3):
    print(f"{r['score']:.3f}  {r['chunk_id']:30s}  {r['text'][:90]}...")


0.676  ug_rulebook::0                  IIT BOMBAY RULES & REGULATIONS FOR UNDERGRADUATE PROGRAMMES (B.Tech./B.S./B.Des./Dual Degr...
0.616  medals_and_prizes::0            IIT BOMBAY - RULES FOR AWARD OF MEDALS AND ACADEMIC PRIZES FOR UG & PG (Approved 260th Sen...
0.612  acad_calendar_2026_27::0        IIT BOMBAY ACADEMIC CALENDAR 2026-2027 (KEY DATES) AUTUMN SEMESTER 2026
- New entrant UG r...


## 7. LLM setup (Gemini API)


In [22]:
import getpass
import google.generativeai as genai

GEMINI_MODEL = "gemini-2.5-flash"

client = None
gemini_api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not gemini_api_key:
    gemini_api_key = getpass.getpass("Enter your Gemini API key: ").strip()

if gemini_api_key:
    os.environ["GEMINI_API_KEY"] = gemini_api_key
    genai.configure(api_key=gemini_api_key)
    client = True
    print("Gemini client ready.")
else:
    print("WARNING: no Gemini API key provided. Set GEMINI_API_KEY (or GOOGLE_API_KEY) before running the generation cells below.")


Gemini client ready.


## 8. Grounded answering

This is the core "no-hallucination" thing:

1. Retrieve top-k chunks for the query.
2. If the best chunk's similarity is below MIN_SIMILARITY, skip the LLM call entirely and return "I don't know" as the retrieval layer itself acts as a gate.
3. Otherwise, inject only the retrieved chunks into the system prompt and instruct the model to answer strictly from the provided context, and to explicitly say it doesn't know if the context doesn't cover the question (e.g. a hostel or scholarship question, which is out of this assistant's academic scope).
4. Return the answer and the list of source chunks actually retrieved, for transparent citation.


In [23]:
SYSTEM_PROMPT = textwrap.dedent('''
You are IITB Insti-Assist, a grounded academic-policy assistant for IIT Bombay.

Rules you MUST follow:
1. Answer ONLY using the information in the "CONTEXT" section below. Do not use any outside
   knowledge about IIT Bombay, even if you believe it to be true.
2. If the CONTEXT does not contain enough information to answer the question, reply exactly:
   "I don't know based on the information I have. This may not be covered in my current
   knowledge base, or you may want to check with the Academic Office directly."
   Do not guess, extrapolate, or fill gaps with general knowledge.
3. When you do answer, be precise and cite specific figures/terms from the context (e.g. grade
   names, credit numbers, dates) rather than vague paraphrases.
4. Keep answers concise (7-8 sentences unless the question needs a list/table).
5. Do not mention that you were "given context" or refer to "the documents" awkwardly and answer
   naturally, as a knowledgeable academic office assistant would.
''').strip()

def build_user_message(query: str, chunks: List[Dict[str, Any]]) -> str:
    context_blocks = []
    for i, c in enumerate(chunks, 1):
        context_blocks.append(f"[Source {i}: {c['title']}]\n{c['text']}")
    context = "\n\n".join(context_blocks)
    return f"CONTEXT:\n{context}\n\nQUESTION: {query}"

def generate_answer(query: str, k: int = 4) -> Dict[str, Any]:
    chunks = retrieve(query, k=k)
    if not chunks or chunks[0]["score"] < MIN_SIMILARITY:
        return {
            "answer": ("I don't know based on the information I have. This may not be covered in "
                       "my current knowledge base, or you may want to check with the Academic "
                       "Office directly."),
            "sources": [],
            "grounded": False,
        }

    if client is None:
        return {
            "answer": "[No GEMINI_API_KEY (or GOOGLE_API_KEY) set -- cannot call the LLM. Showing retrieved context instead:]\n\n"
                       + "\n\n".join(f"- {c['text']}" for c in chunks),
            "sources": chunks,
            "grounded": True,
        }

    user_message = build_user_message(query, chunks)
    model = genai.GenerativeModel(GEMINI_MODEL, system_instruction=SYSTEM_PROMPT)
    response = model.generate_content(
        user_message,
        generation_config={"max_output_tokens": 500, "temperature": 0.2},
    )
    answer_text = getattr(response, "text", None) or "".join(part.text for part in getattr(response, "parts", []) if hasattr(part, "text"))
    return {"answer": answer_text, "sources": chunks, "grounded": True}


## 9. example queries

A mix of **in-scope** questions (should be answered from context) and **out-of-scope** questions(should trigger the "I don't know" refusal), to demonstrate grounded behaviour.


In [24]:
demo_queries = [
    "What CPI category do I need to register for 54 credits in a semester?",
    "What happens if I get caught using my phone during an exam?",
    "What is the difference between a minor and honor program?",
    "When does the Spring 2026 semester end and when are end-semester exams?",
    "What is the mess bill refund policy for hostel students?",
]

for q in demo_queries:
    result = generate_answer(q)
    print("Q:", q)
    print("A:", result["answer"])
    if result["sources"]:
        print("Sources:", ", ".join(sorted(set(s["title"] for s in result["sources"]))))
    print("-" * 100)


Q: What CPI category do I need to register for 54 credits in a semester?
A: To register for 54 credits in a semester, you need to be in Academic Standing Category I. This category requires a Cumulative Performance Index (CPI) of 8 or greater, with no outstanding FR, DX, DR, or W grades in any core course.
Sources: UG Rules & Regulations (Academic Office, updated June 2025)
----------------------------------------------------------------------------------------------------
Q: What happens if I get caught using my phone during an exam?
A: If you are caught using a mobile phone during an exam, the penalty is an FR grade. This is distinct from merely having a mobile phone in your possession (not used) after the exam has begun, which results in the loss of one grade.
Sources: Disciplinary Actions for Academic Malpractice, UG Rules & Regulations (Academic Office, updated June 2025)
----------------------------------------------------------------------------------------------------
Q: What is

## 10. Web interface (Gradio, rendered inline)

Wraps `generate_answer` in a chat UI with:
- **Source display** under every answer (title + similarity score of each retrieved chunk).


In [25]:
import gradio as gr

def format_sources(sources: List[Dict[str, Any]]) -> str:
    if not sources:
        return ""
    lines = ["\n\n---\n**Sources used:**"]
    for s in sources:
        lines.append(f"- *{s['title']}* (chunk `{s['chunk_id']}`, similarity {s['score']:.2f}) — {s['url']}")
    return "\n".join(lines)

def get_last_user_turn(history):
    '''Gradio's ChatInterface passes history as a flat list of {"role","content"} dicts
    (the "messages" format) by default. Walk backwards to find the most recent user turn.'''
    if not history:
        return ""
    for turn in reversed(history):
        if isinstance(turn, dict):
            if turn.get("role") == "user":
                return turn.get("content", "") or ""
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            return turn[0] or ""
    return ""

def chat_fn(message, history):
    last_user = get_last_user_turn(history)
    augmented_query = f"{last_user} {message}".strip() if last_user else message

    result = generate_answer(augmented_query)
    reply = result["answer"] + format_sources(result["sources"])
    return reply

demo = gr.ChatInterface(
    fn=chat_fn,
    title="IITB Insti-Assist — Academic Rules & Calendar",
    description=(
        "Ask about IIT Bombay's grading system, SPI/CPI, registration categories, academic "
        "calendar dates, exam malpractice penalties, or medals/prizes. Answers are grounded in "
        "official Academic Office documents and the assistant will say \"I don't know\" if a "
        "question falls outside that scope."
    ),
    examples=[
        "What grade do I get if I'm caught copying in an exam?",
        "How is CPI calculated?",
        "When is the last date to drop a course in Spring 2026?",
        "What is Category III academic standing?",
        "How much cash prize do 2nd year UG toppers get?",
    ],
)

print("Gradio ChatInterface built.")


Gradio ChatInterface built.


In [26]:
demo.launch(inline=True, height=650)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## 11. Known limitations

- **Data size:** only 5 documents / ~200 chunks. A production assistant would ingest the full UG/PG rule books, department-specific curricula, hostel manuals, and FAQ pages (likely 50+ documents), and re-crawl periodically since the Academic Calendar and Senate-approved rules change most semesters.
- **Chunking:** paragraph+sentence chunking is simple and interpretable but doesn't understand table structure as the Academic Calendar's tabular date ranges lose some of their tabular alignment once flattened to prose. A table-aware parser would improve precision on calendar queries.
- **Embedding model:** all-MiniLM-L6-v2 is fast but general-purpose; a domain-tuned or larger embedding model would likely improve retrieval recall on paraphrased student queries.
- **No re-ranking step:** top-k is taken directly from FAISS cosine similarity. A cross-encoder re-ranker over the top ~20 candidates would help when multiple chunks are topically similar but only one actually answers the question.
- **No automated eval harness:** the "Try it" cell shows a handful of manually chosen examples. With more time I'd build a small labelled test set (in-scope Q/A pairs + deliberately out-of-scope questions) and track precision/recall on the refusal decision, plus answer faithfulness.
